# main.ipynb — Production Analytics Pipeline

Entry point for the Data Factory / Fabric Pipeline.

Loads all modules in the correct order using `%run`, then executes the full KPI report.



## Step 1 — Load functions

In [1]:
import pandas as pd

StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 3, Finished, Available, Finished, False)

In [2]:
%run functions

StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 14, Finished, Available, Finished, True)

In [3]:
csv_path = 'abfss://Pizzatask@onelake.dfs.fabric.microsoft.com/Witside.Lakehouse/Files/dataset.csv'
events = load_events(csv_path)

StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 15, Finished, Available, Finished, False)

[loader] Read 70 rows from abfss://Pizzatask@onelake.dfs.fabric.microsoft.com/Witside.Lakehouse/Files/dataset.csv


## Step 2 — BQ1: Sessions for a Specific Line

In [14]:
LINE_ID = 'gr-np-47'   # change this to query a different line

sessions_df = line_sessions(events, LINE_ID)
display(sessions_df)
sessions_spark = sessions_df.copy()
sessions_spark['duration_seconds'] = sessions_df['duration'].dt.total_seconds()
sessions_spark = sessions_spark.drop(columns=['duration'])
# Convert pandas → Spark DataFrame
spark_ses = spark.createDataFrame(sessions_spark)


# Write to Lakehouse as Delta table
spark_ses.write.format("delta").mode("overwrite").saveAsTable("Results_Q1")


StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c4667151-5a6b-452b-bf80-63ec69a1a0ca)

## Step 3 — BQ2: Floor Uptime / Downtime

In [5]:
summary = floor_uptime_downtime(events)
display(summary[['production_line_id','total_uptime','total_downtime','uptime_seconds','downtime_seconds']])

StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 363a2810-24dd-4e87-a4ec-123603a42d09)

In [6]:
# Convert pandas → Spark DataFrame
spark_df = spark.createDataFrame(summary[['production_line_id','uptime_seconds','downtime_seconds']])


# Write to Lakehouse as Delta table
spark_df.write.format("delta").mode("overwrite").saveAsTable("Results_Q2")

StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 18, Finished, Available, Finished, False)

## Step 4 — BQ3: Line with Most Downtime

In [10]:
worst = most_downtime_line(events)
worst_df = pd.DataFrame([{
    'production_line_id': worst['production_line_id'],
    'downtime_hms':       worst['downtime_hms'],
    'downtime_seconds':   worst['downtime_seconds'],
}])
# Convert pandas → Spark DataFrame
spark_w = spark.createDataFrame(worst_df)


# Write to Lakehouse as Delta table
spark_w.write.format("delta").mode("overwrite").saveAsTable("Results_Q3")

print(f"Line     : {worst['production_line_id']}")
print(f"Downtime : {worst['downtime_hms']}")
print(f"Seconds  : {worst['downtime_seconds']:,.0f} s")

StatementMeta(, ea6d73b8-735c-4d0e-b532-c8cdf805bcaf, 22, Finished, Available, Finished, False)

Line     : gr-np-47
Downtime : 00h 56m 40s
Seconds  : 3,400 s


## Step 5 — Full Formatted Report

In [21]:
print_report(events, line_id=LINE_ID)

StatementMeta(, 2064b68d-5059-4951-b4fa-4516978667ea, 92, Finished, Available, Finished, False)

  BQ1 - Session table for: gr-np-47
      start_timestamp      stop_timestamp        duration duration_hms
0 2020-10-07 01:33:20 2020-10-07 02:03:20 0 days 00:30:00  00h 30m 00s
1 2020-10-07 02:15:02 2020-10-07 04:15:02 0 days 02:00:00  02h 00m 00s
2 2020-10-07 05:00:00 2020-10-07 05:55:17 0 days 00:55:17  00h 55m 17s

  BQ2 - Total uptime / downtime per line
  production_line_id total_uptime total_downtime uptime_seconds downtime_seconds
0           gr-np-22  00h 00m 00s    00h 00m 00s              0                0
1           gr-np-47  03h 25m 17s    00h 56m 40s         12,317            3,400
2        FLOOR TOTAL  03h 25m 17s    00h 56m 40s         12,317            3,400

  BQ3 - Line with the most downtime
  Line ID       : gr-np-47
  Total downtime: 00h 56m 40s
  (seconds)     : 3,400 s
